# Tools and Agents (open source): LCEL

### Outline
- A simple chain: `prompt | model | parser`
- A more complex chain: a `RunnableMap` feeding a toy retriever into a RAG-style prompt
- `bind`: attaching tools to a model at runtime
- Fallbacks: `runnable.with_fallbacks([...])`
- The `invoke` / `batch` / `stream` interface

Open-source recode of `03-Functions-Tools-and-Agents-with-LangChain/L2-lcel-student.ipynb`, using
`ChatOllama` in place of `ChatOpenAI`. Same LCEL concepts as the original.

**Retriever note**: the original builds a retriever with `OpenAIEmbeddings` + `DocArrayInMemorySearch`.
The Ollama cloud account behind this project's `.env` returns "unauthorized" for the embeddings
endpoint, so the "more complex chain" section below uses a tiny hand-rolled keyword matcher
instead. It plugs into the same `RunnableMap` shape; swap in `OllamaEmbeddings` +
`InMemoryVectorStore` if you have a local embedding model pulled and point `base_url` at a local
Ollama instance.

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`tools_and_agent`](../../tools_and_agent/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [ ]:
import sys
from pathlib import Path

# common.py / tracing.py live in tools_and_agent/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../tools_and_agent").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

import json

from common import get_model, strip_json_fence, traced
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableMap

## Simple chain

`prompt | model | output_parser` - the same three-stage pipe every chain in this course builds
on.

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")
model = get_model()
output_parser = StrOutputParser()

chain = prompt | model | output_parser

print(chain.invoke({"topic": "bears"}, config=traced("L2: simple joke chain")))

## More complex chain

A `RunnableMap` builds the `{context, question}` dict a RAG prompt needs, running the retriever
and passing the question through in parallel.

In [ ]:
corpus = ["harrison worked at kensho", "bears like to eat honey"]


def toy_retriever(question: str) -> list[str]:
    """Keyword-overlap 'retriever' standing in for a real vector store (see intro note above)."""
    scored = sorted(
        corpus,
        key=lambda doc: len(set(doc.lower().split()) & set(question.lower().split())),
        reverse=True,
    )
    return scored[:2]


template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
rag_prompt = ChatPromptTemplate.from_template(template)

rag_inputs = RunnableMap(
    {
        "context": lambda x: toy_retriever(x["question"]),
        "question": lambda x: x["question"],
    }
)

print(rag_inputs.invoke({"question": "where did harrison work?"}))

In [ ]:
rag_chain = rag_inputs | rag_prompt | model | output_parser

print(rag_chain.invoke({"question": "where did harrison work?"}, config=traced("L2: RAG-style chain")))

## Bind: attaching tools at runtime

`.bind_tools([...])` attaches tool schemas to a model without changing the prompt - the model
decides whether/which tool to call based on the input.

In [ ]:
weather_tool = {
    "name": "weather_search",
    "description": "Search for weather given an airport code",
    "parameters": {
        "type": "object",
        "properties": {
            "airport_code": {
                "type": "string",
                "description": "The airport code to get the weather for",
            },
        },
        "required": ["airport_code"],
    },
}

bind_prompt = ChatPromptTemplate.from_messages([("human", "{input}")])
bound_model = get_model().bind_tools([weather_tool])
runnable = bind_prompt | bound_model

print(runnable.invoke({"input": "what is the weather in sf"}, config=traced("L2: bind single tool")).tool_calls)

With two tools bound, the model has to pick the right one.

In [ ]:
sports_tool = {
    "name": "sports_search",
    "description": "Search for news of recent sport events",
    "parameters": {
        "type": "object",
        "properties": {
            "team_name": {
                "type": "string",
                "description": "The sports team to search for",
            },
        },
        "required": ["team_name"],
    },
}

bound_model = get_model().bind_tools([weather_tool, sports_tool])
runnable = bind_prompt | bound_model

print(
    runnable.invoke(
        {"input": "how did the patriots do yesterday?"}, config=traced("L2: bind two tools")
    ).tool_calls
)

## Fallbacks

The original pairs a completion-style model (prone to producing non-JSON text around its answer)
with a chat model as a fallback. Here the "flaky" step is a plain chat model piped straight into
`json.loads` - it tends to add prose around the JSON and fails to parse. The fallback swaps in
`format="json"` plus `strip_json_fence` (this model still wraps JSON in ` ```json ` fences even in
JSON mode). The pattern itself is the point: `runnable.with_fallbacks([other_runnable])`.

In [ ]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"

flaky_model = get_model()  # no format="json" -> usually wraps the answer in prose
flaky_chain = flaky_model | StrOutputParser() | json.loads

reliable_model = get_model(format="json")
reliable_chain = reliable_model | StrOutputParser() | strip_json_fence | json.loads

final_chain = flaky_chain.with_fallbacks([reliable_chain])

print(final_chain.invoke(challenge, config=traced("L2: fallback chain")))

## Interface: invoke / batch / stream

Every LCEL runnable exposes the same three methods, regardless of what it's built from.

In [ ]:
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
model = get_model()
chain = prompt | model | StrOutputParser()

print(chain.invoke({"topic": "bears"}, config=traced("L2: interface invoke")))

In [ ]:
print(chain.batch([{"topic": "bears"}, {"topic": "frogs"}], config=traced("L2: interface batch")))

In [ ]:
for chunk in chain.stream({"topic": "bears"}, config=traced("L2: interface stream")):
    print(chunk, end="", flush=True)
print()